# Tool 4 : Retrieval Embedding Model (from scratch)

A small contrastive embedding network trained on `(finding_label, reference_text)` pairs.
At inference the agent embeds a query like `"CNV finding"` and retrieves the closest reference snippets.

**Architecture:** Two-tower model. Both query and document towers share a small transformer encoder
(4 layers, 128-dim hidden, ~800K params). Trained with NT-Xent (in-batch negatives).

**Training data:** Curated radiology reference corpus assembled in Section 1 below.
You can extend this with any OCT / radiology text you have access to.

**Pipeline:**
```
Query string → tokenize → SharedEncoder → L2-normalize → cosine sim vs corpus embeddings → top-k snippets
```

## 0. Imports

In [ ]:
import random, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from collections import defaultdict
from pathlib import Path

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Reference Corpus
Each entry is a `(label, text)` pair. Extend with more snippets for better coverage.
Labels must match `CLASS_NAMES` from your classifier: `CNV | DME | DRUSEN | NORMAL`.

In [ ]:

CORPUS = [
    # CNV — Choroidal Neovascularization
    ("CNV", "Choroidal neovascularization (CNV) refers to the growth of new blood vessels from the choroid through Bruch's membrane into the sub-retinal pigment epithelium or subretinal space."),
    ("CNV", "On OCT, CNV typically presents as a hyperreflective irregular lesion beneath the RPE, often associated with subretinal fluid and disruption of the ellipsoid zone."),
    ("CNV", "Subretinal fluid appearing as a hyporeflective space between the neurosensory retina and RPE is a hallmark of active CNV."),
    ("CNV", "Type 1 CNV (occult) grows beneath the RPE, producing a pigment epithelial detachment (PED) visible as a dome-shaped elevation of the RPE on OCT."),
    ("CNV", "Type 2 CNV (classic) grows through the RPE into the subretinal space, appearing as a hyperreflective lesion above the RPE layer."),
    ("CNV", "Intraretinal fluid cysts in combination with sub-RPE material strongly suggest neovascular AMD with CNV activity."),
    ("CNV", "Anti-VEGF therapy is the standard of care for CNV, targeting vascular endothelial growth factor to suppress new vessel growth."),
    ("CNV", "Irregular RPE elevation with overlying intraretinal or subretinal fluid on OCT is highly suggestive of CNV requiring treatment."),

    # DME — Diabetic Macular Edema
    ("DME", "Diabetic macular edema (DME) is characterized by intraretinal fluid accumulation in the macula secondary to breakdown of the inner blood-retinal barrier."),
    ("DME", "OCT in DME shows retinal thickening with hyporeflective intraretinal cystic spaces, most prominent in the outer nuclear and inner nuclear layers."),
    ("DME", "Center-involved DME refers to edema affecting the central 1mm subfield of the macula, which is the primary target for treatment decisions."),
    ("DME", "Hard exudates appear as hyperreflective foci with posterior shadowing on OCT and indicate lipid deposition from chronic vascular leakage in DME."),
    ("DME", "Diffuse retinal thickening without distinct cysts, or sponge-like swelling, is a common OCT pattern in early or mild DME."),
    ("DME", "Vitreomacular traction may exacerbate DME, with OCT showing posterior hyaloid attachment and tractional elevation of the retina."),
    ("DME", "Anti-VEGF injections and corticosteroid implants are first-line treatments for center-involving DME with vision loss."),
    ("DME", "Ellipsoid zone (IS/OS) disruption in DME on OCT correlates with photoreceptor damage and predicts worse visual acuity outcomes."),

    # DRUSEN
    ("DRUSEN", "Drusen are extracellular deposits beneath the RPE, composed of lipids, proteins, and cellular debris, and are the hallmark of early and intermediate AMD."),
    ("DRUSEN", "On OCT, soft drusen appear as dome-shaped, medium-reflective elevations of the RPE with a distinct base and apex, without overlying retinal fluid."),
    ("DRUSEN", "Hard drusen are small, discrete, and hyperreflective on OCT; they carry low risk of progression to advanced AMD compared to soft or confluent drusen."),
    ("DRUSEN", "Drusen volume and area on OCT are quantitative biomarkers used to stratify progression risk from intermediate to late AMD."),
    ("DRUSEN", "Subretinal drusenoid deposits (reticular pseudodrusen) appear as hyperreflective material above the RPE and are associated with increased AMD progression risk."),
    ("DRUSEN", "Drusen regression — a reduction in drusen volume — paradoxically precedes RPE atrophy (geographic atrophy) in advanced dry AMD."),
    ("DRUSEN", "The presence of large soft drusen (>125 µm) with pigmentary changes increases the 5-year risk of progression to neovascular or atrophic AMD substantially."),
    ("DRUSEN", "Outer retinal tubulations adjacent to drusen on OCT indicate photoreceptor degeneration and RPE atrophy in geographic atrophy."),

    # NORMAL
    ("NORMAL", "A normal OCT macula shows a smooth foveal contour with a central pit, distinct retinal layers, an intact ellipsoid zone, and no fluid or drusen."),
    ("NORMAL", "The normal foveal avascular zone appears as a central depression on OCT with thinning of the inner retinal layers and preservation of the outer retinal bands."),
    ("NORMAL", "In a healthy retina, the RPE/Bruch's complex appears as a continuous, uniformly hyperreflective band without elevations or disruptions."),
    ("NORMAL", "Normal central subfield thickness (CST) on OCT is approximately 250–270 µm in women and 270–290 µm in men on Spectralis OCT."),
    ("NORMAL", "The ellipsoid zone (previously called the IS/OS junction) appears as a bright, continuous hyperreflective line in the normal outer retina."),
    ("NORMAL", "Normal OCT shows no intraretinal cysts, no subretinal fluid, no RPE elevations, and no hyperreflective foci indicative of pathology."),
    ("NORMAL", "The inner segment / outer segment (IS/OS) junction and the cone outer segment tip (COST) line are both visible and intact in a normal healthy macula."),
    ("NORMAL", "Age-related thinning of the retinal nerve fiber layer (RNFL) on OCT is a normal finding and should be distinguished from glaucomatous loss."),
]

print(f'Corpus: {len(CORPUS)} entries across {len(set(l for l,_ in CORPUS))} classes')
for label in ['CNV', 'DME', 'DRUSEN', 'NORMAL']:
    n = sum(1 for l, _ in CORPUS if l == label)
    print(f'  {label}: {n} snippets')

In [ ]:
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted
import time

# 1. Configuration
import os
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-2.0-flash")


def paraphrase(label: str, text: str, n: int = 2) -> list[tuple]:
    prompt = (
        f"Write {n} different paraphrases of this clinical ophthalmology text. "
        f"Keep the medical meaning exact. Return only the paraphrases, one per line, no numbering.\n\n{text}"
    )
    resp = model.generate_content(prompt)
    lines = [l.strip() for l in resp.text.strip().split('\n') if l.strip()]
    return [(label, line) for line in lines[:n]]

# Augmentation Loop with Safety Nets
augmented = []
print("Starting augmentation process...\n")

for label, text in CORPUS:
    success = False
    
    while not success:
        try:
            new_pairs = paraphrase(label, text, n=2)
            augmented.extend(new_pairs)
            print(f"  [SUCCESS] {label}: +{len(new_pairs)} paraphrases added.")
            
            success = True 
            time.sleep(3) # A brief pause to keep the API happy
            
        except ResourceExhausted:
            print("  [RATE LIMIT HIT] Sleeping for 60 seconds before retrying...")
            time.sleep(60)
            
        except Exception as e:
            print(f"  [ERROR] Something went wrong on {label}: {e}")
            break # Skip this one and move to the next if it's a structural error

CORPUS_EXPANDED = CORPUS + augmented

print(f'\nOriginal : {len(CORPUS)} entries')
print(f'Augmented: {len(augmented)} entries')
print(f'Total    : {len(CORPUS_EXPANDED)} entries')

## 2. Character-level Tokenizer
Tiny vocabulary (char + subword BPE is overkill for this corpus size).
Simple word-level tokenizer with a fixed vocab built from the corpus.

In [ ]:
class WordTokenizer:
    PAD, UNK = '<PAD>', '<UNK>'

    def __init__(self, texts: list[str], max_len: int = 64):
        self.max_len = max_len
        words = [w.lower().strip('.,();:"\'-') for t in texts for w in t.split()]
        vocab = [self.PAD, self.UNK] + sorted(set(words))
        self.w2i = {w: i for i, w in enumerate(vocab)}
        self.vocab_size = len(vocab)
        print(f'Vocab size: {self.vocab_size}')

    def encode(self, text: str) -> torch.Tensor:
        tokens = [text.lower().strip('.,();:"\'-') for text in text.split()]
        ids = [self.w2i.get(t, 1) for t in tokens]  # 1 = UNK
        # Pad or truncate
        ids = ids[:self.max_len] + [0] * max(0, self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)


all_texts = [t for _, t in CORPUS] + [l for l, _ in CORPUS]
tokenizer = WordTokenizer(all_texts, max_len=64)

## 3. Encoder Architecture
Small transformer: embedding → 4× TransformerEncoder layers → mean pool → L2 normalize.

In [ ]:
class SmallEncoder(nn.Module):
    """
    Shared encoder tower used for both queries and documents.
    ~800K parameters.

    Architecture:
        token embedding (vocab → d_model)
        → positional encoding
        → N × TransformerEncoderLayer
        → mean-pool over sequence
        → linear projection to embed_dim
        → L2 normalize
    """
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 4,
        dim_ff: int = 256,
        embed_dim: int = 64,
        max_len: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)           # learned positional
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True,  # pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.proj = nn.Linear(d_model, embed_dim, bias=False)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.tok_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """input_ids: (B, L) → embeddings: (B, embed_dim) L2-normalized."""
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, -1)
        padding_mask = (input_ids == 0)                          # True where PAD

        x = self.tok_emb(input_ids) + self.pos_emb(positions)   # (B, L, d_model)
        x = self.transformer(x, src_key_padding_mask=padding_mask)

        # Mean pool over non-padding tokens
        mask = (~padding_mask).float().unsqueeze(-1)             # (B, L, 1)
        x = (x * mask).sum(1) / mask.sum(1).clamp(min=1)        # (B, d_model)

        x = self.proj(x)                                         # (B, embed_dim)
        return F.normalize(x, dim=-1)                            # L2 normalize


encoder = SmallEncoder(vocab_size=tokenizer.vocab_size).to(DEVICE)
n_params = sum(p.numel() for p in encoder.parameters())
print(f'Encoder parameters: {n_params:,} ({n_params/1e6:.2f}M)')

## 4. Training Data : Contrastive Pairs
Positive pairs: `(finding_label, reference_text)` with the same label.
Negatives: in-batch (all other pairs in the batch).

In [ ]:
class ContrastivePairDataset(Dataset):
    """
    Each item is a (query_ids, doc_ids) positive pair.
    Queries are constructed as: '<LABEL> finding' and '<LABEL> <synonym>'.
    Documents are the reference text snippets.
    Negatives are handled in-batch inside the loss function.
    """
    # Richer query templates so the encoder sees varied phrasing
    QUERY_TEMPLATES = [
        "{label} finding",
        "{label} OCT appearance",
        "{label} diagnosis",
        "{label} retinal features",
        "{label} clinical signs",
        "what does {label} look like on OCT",
        "{label} imaging characteristics",
    ]

    def __init__(self, corpus: list[tuple[str, str]], tokenizer: WordTokenizer):
        self.tokenizer = tokenizer
        self.pairs = []  # (query_str, doc_str)
        for label, text in corpus:
            for tmpl in self.QUERY_TEMPLATES:
                self.pairs.append((tmpl.format(label=label), text))
        random.shuffle(self.pairs)
        print(f'Training pairs: {len(self.pairs)}')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, d = self.pairs[idx]
        return self.tokenizer.encode(q), self.tokenizer.encode(d)


dataset = ContrastivePairDataset(CORPUS, tokenizer)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

## 5. NT-Xent Loss (in-batch negatives)
Standard contrastive loss: pull positive (q, d+) pairs together,
push against all other docs in the batch as negatives.

In [ ]:
def nt_xent_loss(q_emb: torch.Tensor, d_emb: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """
    NT-Xent with in-batch negatives.
    q_emb, d_emb: (B, embed_dim), already L2-normalized.
    Positive pairs are on the diagonal (q_i ↔ d_i).
    All off-diagonal (q_i, d_j≠i) are negatives.
    """
    # (B, B) similarity matrix
    sim = torch.matmul(q_emb, d_emb.T) / temperature   # (B, B)
    labels = torch.arange(sim.size(0), device=sim.device)  # 0,1,...,B-1
    # Cross entropy: each query should match its own doc (diagonal)
    loss_q2d = F.cross_entropy(sim, labels)             # query→doc direction
    loss_d2q = F.cross_entropy(sim.T, labels)           # doc→query direction
    return (loss_q2d + loss_d2q) / 2


print('NT-Xent loss defined.')

## 6. Training

In [ ]:
EPOCHS_EMB   = 80
LR_EMB       = 3e-4
TEMPERATURE  = 0.07
CKPT_EMB     = 'retrieval_encoder.pth'

optimizer = AdamW(encoder.parameters(), lr=LR_EMB, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_EMB, eta_min=1e-5)

best_loss = float('inf')
history   = []

for ep in range(1, EPOCHS_EMB + 1):
    encoder.train()
    ep_loss = 0.0
    for q_ids, d_ids in loader:
        q_ids, d_ids = q_ids.to(DEVICE), d_ids.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        q_emb = encoder(q_ids)
        d_emb = encoder(d_ids)
        loss  = nt_xent_loss(q_emb, d_emb, TEMPERATURE)
        loss.backward()
        nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()

    avg = ep_loss / len(loader)
    history.append(avg)
    if avg < best_loss:
        best_loss = avg
        torch.save(encoder.state_dict(), CKPT_EMB)
    if ep % 10 == 0 or ep == 1:
        print(f'Ep {ep:03d}/{EPOCHS_EMB} | loss {avg:.4f} | lr {scheduler.get_last_lr()[0]:.2e}'
              + (' ★' if avg == best_loss else ''))

print(f'\nBest loss: {best_loss:.4f} — saved to {CKPT_EMB}')

## 7. Build Corpus Index
Embed all reference documents once and store them.

In [ ]:
# Load best checkpoint
encoder.load_state_dict(torch.load(CKPT_EMB, map_location=DEVICE))
encoder.eval()

@torch.no_grad()
def embed_texts(texts: list[str]) -> torch.Tensor:
    """Embed a list of strings → (N, embed_dim) tensor."""
    ids = torch.stack([tokenizer.encode(t) for t in texts]).to(DEVICE)
    return encoder(ids).cpu()


# Embed all corpus documents
corpus_texts  = [text  for _, text  in CORPUS]
corpus_labels = [label for label, _ in CORPUS]
corpus_embs   = embed_texts(corpus_texts)  # (N, embed_dim)

print(f'Corpus embeddings: {corpus_embs.shape}  (N documents × embed_dim)')

## 8. Retrieval Function

In [ ]:
@torch.no_grad()
def retrieve(
    query: str,
    top_k: int = 3,
    label_filter: str | None = None,
) -> list[dict]:
    """
    Retrieve the top-k most relevant reference snippets for a query.

    Args:
        query:        Natural language query, e.g. 'CNV finding' or 'subretinal fluid'.
        top_k:        Number of results to return.
        label_filter: If set (e.g. 'CNV'), only retrieve from that finding class.

    Returns:
        List of dicts with keys: rank, label, text, score
    """
    q_emb = embed_texts([query])           # (1, embed_dim)
    scores = (corpus_embs @ q_emb.T).squeeze(1)  # cosine sim, (N,)

    if label_filter:
        mask = torch.tensor([l == label_filter for l in corpus_labels])
        scores = scores.masked_fill(~mask, -1.0)

    top_idx = scores.argsort(descending=True)[:top_k]
    return [
        {
            'rank':  rank + 1,
            'label': corpus_labels[i],
            'text':  corpus_texts[i],
            'score': float(scores[i]),
        }
        for rank, i in enumerate(top_idx.tolist())
    ]


# Quick test
print(' Test: "CNV finding"')
for r in retrieve('CNV finding', top_k=3):
    print(f"  [{r['rank']}] {r['label']} (score={r['score']:.3f}): {r['text'][:80]}...")

print('\nTest: "fluid beneath retina"')
for r in retrieve('fluid beneath retina', top_k=3):
    print(f"  [{r['rank']}] {r['label']} (score={r['score']:.3f}): {r['text'][:80]}...")

## 9. Retrieval Evaluation
Precision@1 and Precision@3 on a hand-labeled query set.
A retrieval is correct if the top-k results include any snippet with the correct label.

In [ ]:
# Evaluation query set — (query_string, correct_label)
# Each query should be answered by at least one snippet of the correct label.
EVAL_QUERIES = [
    ("CNV finding",                         "CNV"),
    ("choroidal neovascularization OCT",     "CNV"),
    ("subretinal fluid appearance",          "CNV"),
    ("pigment epithelial detachment",        "CNV"),
    ("DME finding",                          "DME"),
    ("diabetic macular edema OCT",           "DME"),
    ("intraretinal cysts diabetic",          "DME"),
    ("hard exudates retina",                 "DME"),
    ("DRUSEN finding",                       "DRUSEN"),
    ("drusen AMD macular degeneration",      "DRUSEN"),
    ("RPE elevation dome shaped",            "DRUSEN"),
    ("soft drusen OCT appearance",           "DRUSEN"),
    ("NORMAL finding",                       "NORMAL"),
    ("normal healthy macula OCT",            "NORMAL"),
    ("intact ellipsoid zone no fluid",       "NORMAL"),
    ("foveal pit normal retina",             "NORMAL"),
]

def precision_at_k(queries: list[tuple[str, str]], k: int) -> float:
    hits = 0
    for query, correct_label in queries:
        results = retrieve(query, top_k=k)
        retrieved_labels = [r['label'] for r in results]
        if correct_label in retrieved_labels:
            hits += 1
    return hits / len(queries)


p1 = precision_at_k(EVAL_QUERIES, k=1)
p3 = precision_at_k(EVAL_QUERIES, k=3)
print(f'Precision@1 : {p1*100:.1f}%  ({int(p1*len(EVAL_QUERIES))}/{len(EVAL_QUERIES)} correct)')
print(f'Precision@3 : {p3*100:.1f}%  ({int(p3*len(EVAL_QUERIES))}/{len(EVAL_QUERIES)} correct)')

# Per-class breakdown
print('\nPer-class P@1:')
for label in ['CNV', 'DME', 'DRUSEN', 'NORMAL']:
    cls_q = [(q, l) for q, l in EVAL_QUERIES if l == label]
    p = precision_at_k(cls_q, k=1)
    print(f'  {label:8s}: {p*100:.0f}%')

## 10. Training curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history)+1), history)
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('Retrieval Encoder — Training Loss')
plt.tight_layout()
plt.savefig('tool4_training_curve.png', dpi=150)
plt.show()

# Embedding space visualization (PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(corpus_embs.numpy())
colors = {'CNV': 'red', 'DME': 'blue', 'DRUSEN': 'green', 'NORMAL': 'gray'}

plt.figure(figsize=(8, 6))
for label in ['CNV', 'DME', 'DRUSEN', 'NORMAL']:
    idxs = [i for i, l in enumerate(corpus_labels) if l == label]
    plt.scatter(emb_2d[idxs, 0], emb_2d[idxs, 1], label=label, color=colors[label], s=80, alpha=0.8)
plt.legend(); plt.title('Corpus Embeddings — PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('tool4_embedding_space.png', dpi=150)
plt.show()